# DataMate – Nấc A (OmniSQL-7B → SQL → đáp án)
**Chuẩn bị trên Kaggle (1 lần):**
1. Notebook → Settings: **Accelerator = GPU T4 x2**, **Internet = On** (để tải model từ Hugging Face).
2. Tạo Dataset `datamate-code` từ thư mục `datamate/` (file zip tôi gửi), và Dataset `zaic-data` chứa các thư mục `train/` và `public_test/` (đúng cấu trúc `<split>/data/<biz>/...`, `rulebook.md`, `questions.jsonl`...).
3. Add cả hai dataset vào notebook, sửa 2 biến `CODE`, `DATA` bên dưới cho khớp tên.

In [ ]:
import os, shutil
CODE = "/kaggle/input/datamate-code"      # <-- sửa cho khớp
DATA = "/kaggle/input/zaic-data"          # <-- sửa cho khớp
shutil.copytree(CODE, "/kaggle/working/datamate", dirs_exist_ok=True)
os.chdir("/kaggle/working/datamate")
print(os.listdir("."), os.listdir(DATA))

## Bước 1 – Cài vLLM (nhanh hơn nhiều so với transformers)
Nếu lỗi cài đặt, bỏ qua và dùng `--backend hf` ở các bước dưới.

In [ ]:
!pip install -q vllm 2>&1 | tail -2

## Bước 2 – Chạy thử 100 câu đầu của TRAIN và chấm điểm
Train có đáp án nên chấm được ngay bằng `score.py`. Chạy `--limit 100` trước để chắc chắn mọi thứ hoạt động rồi mới chạy cả tập.

In [ ]:
!python run_a.py --split_dir {DATA}/train --out /kaggle/working/train_pred.jsonl \
    --backend vllm --tp 2 --limit 100 --work_dir /kaggle/working/work

In [ ]:
!python score.py --pred /kaggle/working/train_pred.jsonl \
    --ref {DATA}/train/answers.jsonl --questions {DATA}/train/questions.jsonl --only_run

> Cờ `--only_run` chỉ chấm các câu đã chạy (100 câu đầu). Khi quen rồi, bỏ `--limit` và `--only_run` để chạy và chấm toàn bộ train.

## Bước 3 – Chạy PUBLIC TEST và tạo `submission.jsonl` để nộp

In [ ]:
!python run_a.py --split_dir {DATA}/public_test --out /kaggle/working/submission.jsonl \
    --backend vllm --tp 2 --work_dir /kaggle/working/work

In [ ]:
import json, collections
rows=[json.loads(l) for l in open('/kaggle/working/submission.jsonl',encoding='utf-8')]
print(len(rows), collections.Counter(r['type'] for r in rows))

Tải `/kaggle/working/submission.jsonl` về rồi nộp ở mục **Submission** của Zalo AI Challenge (tối đa 5 lượt/ngày).
Log từng câu nằm ở `/kaggle/working/logs/logs.jsonl` (id, đáp án, thời gian, SQL, lỗi) – đề yêu cầu có log kiểu này khi nộp Docker.

**Nếu vLLM không cài được / hết bộ nhớ:** thay `--backend vllm --tp 2` bằng `--backend hf --batch_size 2` (chậm hơn). Trên Colab T4 (1 GPU) dùng `--backend hf --quant 4bit`.